### `test_etd_solvers.ipynb` 
*Created: Sept 22, 2026* <br/>
Notebook for testing some Exponential Time Differencing (ETD) solvers by comparing them against trusted reference solvers from `OrdinaryDiffEq.jl`. We also test against exact solutions, when available.

In [2]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
using OrdinaryDiffEqExponentialRK, SciMLOperators 
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [16]:
#Import ETD test problems and solvers
@nbinclude("etd_test_problems.ipynb")

#Use desired plotting defaults for Makie 
@nbinclude("../../../../../../set_makie_defaults.ipynb")

In [ ]:
function compare_solutions(prob::SemilinearODEProblem, reference_alg::OrdinaryDiffEqAlgorithm, custom_alg::A; dt::Real = 0.01) where {A}
    """    
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg :: algorithm that I wrote, that I'm testing 
    """
  
    # @unpack f, u0, tspan, p = prob

   
    # custom_sol = custom_alg(f, u0, tspan, p; dt = dt)
    # ref_sol = solve(ODEProblem(f, u0, tspan, p), reference_alg; adaptive = false, dt = dt, saveat = custom_sol.t)

    # #STEP 3: Compute difference between reference solution and custom solution 
    # u_custom = custom_sol.u
    # u_ref = ref_sol.u
    # max_l2_error = maximum(norm.(u_ref .- u_custom))

    # return (reference_sol = ref_sol, custom_sol = custom_sol, max_l2_error = max_l2_error)     
end 

In [ ]:
results = run_tests(; title = "Runge-Kutta Tests") do suite

    reference_alg = Euler()
    custom_alg = euler

    # reference_alg = RK4()
    # custom_alg = rk4
    
    tol = 1e-8
    dt = 0.01

    test!(suite, "Exponential") do
        @test compare_solutions(exp_growth, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Sinusoid") do
        @test compare_solutions(sinusoid, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Damped Oscillator") do
        @test compare_solutions(damped_oscillator, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Lotka-Volterra") do
        @test compare_solutions(lotka_volterra, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Lorenz System") do
        @test compare_solutions(lorenz_system, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end
end;